In [2]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [4]:
import sys
import importlib

# Add the Silver layer directory to sys.path BEFORE importing
silver_dir = "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver"
if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)

# Now import (and reload to pick up edits)
import utils_silver
from utils_silver import *

importlib.reload(utils_silver)

paths_bronze, paths_silver, paths_gold = utils_silver.build_paths(storage_account1)
print(sorted(paths_silver.keys()))
print("Imported utils_silver from", utils_silver.__file__)

✅ utils_silver.py loaded
✅ utils_silver.py loaded
['_metrics', '_quarantine', '_ref_dim_channel', '_ref_dim_product_line', '_reference', 'claims', 'members', 'policies', 'providers']
Imported utils_silver from /Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver/utils_silver.py


# 🟡 Cell 1 — Imports, paths, configs

In [6]:
# Cell 1 — Imports, paths, configs

from pyspark.sql import functions as F
from pyspark.sql.types import *
from utils_silver import build_paths, write_metric  # reuse same utils

# Storage account you already use
STORAGE_ACCOUNT = "clientdatastorage"

# Get bronze / silver paths (we'll also reuse silver for metrics)
paths_bronze, paths_silver, paths_gold = build_paths(STORAGE_ACCOUNT)

SILVER_MEMBERS_PATH = paths_silver["members"]

# For Gold we keep using the container "golddata" 
GOLD_BASE = f"abfss://golddata@{STORAGE_ACCOUNT}.dfs.core.windows.net"
GOLD_FACT_MEMBERS_PATH = f"{GOLD_BASE}/fact_members"

DB_GOLD = "bupa_gold"
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_GOLD}")

print("SILVER_MEMBERS_PATH    :", SILVER_MEMBERS_PATH)
print("GOLD_FACT_MEMBERS_PATH :", GOLD_FACT_MEMBERS_PATH)
print("DB_GOLD                :", DB_GOLD)


SILVER_MEMBERS_PATH    : abfss://silverdata@clientdatastorage.dfs.core.windows.net/members
GOLD_FACT_MEMBERS_PATH : abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_members
DB_GOLD                : bupa_gold


# 🟡 Cell 2 — Read Silver members → base fact

In [7]:
# Cell 2 — Read Silver members → base fact

members_silver = spark.read.format("delta").load(SILVER_MEMBERS_PATH)

print("Silver members rowcount:", members_silver.count())
members_silver.printSchema()

# Base fact selection (rename lightly for clarity if you like)
fact_base = (
    members_silver.select(
        F.col("Member_ID"),
        F.col("Policy_ID"),
        F.col("Age").cast("int").alias("Age"),
        F.col("Gender").alias("Gender"),
        F.col("BMI").cast("double").alias("BMI"),
        F.col("Smoker").alias("Smoker_Flag"),
        F.col("Chronic_Disease"),
        F.col("Employment_Status"),
        F.col("Region")
    )
)

print("fact_base rowcount:", fact_base.count())
fact_base.show(5, truncate=False)


25/12/08 23:08:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Silver members rowcount: 381109
root
 |-- Member_ID: string (nullable = true)
 |-- Policy_ID: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- BMI: double (nullable = true)
 |-- Smoker: string (nullable = true)
 |-- Chronic_Disease: string (nullable = true)
 |-- Employment_Status: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- dq_age_valid: integer (nullable = true)
 |-- dq_bmi_valid: integer (nullable = true)
 |-- Age_Band: string (nullable = true)
 |-- BMI_Category: string (nullable = true)
 |-- Risk_Segment: string (nullable = true)

fact_base rowcount: 381109


+------------+----------+---+------+------------------+-----------+---------------+-----------------+------+
|Member_ID   |Policy_ID |Age|Gender|BMI               |Smoker_Flag|Chronic_Disease|Employment_Status|Region|
+------------+----------+---+------+------------------+-----------+---------------+-----------------+------+
|MEM_00000008|P_00000008|56 |F     |23.140940890221643|Y          |Asthma         |Student          |280   |
|MEM_00001714|P_00001714|71 |M     |27.963719840043765|N          |None           |Employed         |360   |
|MEM_00001746|P_00001746|75 |M     |30.541100957674118|N          |Diabetes       |Student          |80    |
|MEM_00001967|P_00001967|37 |F     |25.982850362358867|Y          |Hypertension   |Employed         |280   |
|MEM_00001995|P_00001995|66 |M     |32.6294573451364  |N          |Asthma         |Employed         |360   |
+------------+----------+---+------+------------------+-----------+---------------+-----------------+------+
only showing top 5 

# 🟡 Cell 3 — Feature engineering: age & BMI bands, smoker status

In [8]:
# Cell 3 — Feature engineering: Age_Band, BMI_Band, Smoker_Status

fact_enriched_1 = (
    fact_base
    # Age banding
    .withColumn(
        "Age_Band",
        F.when(F.col("Age").isNull(), "Unknown")
         .when(F.col("Age") < 18, "<18")
         .when(F.col("Age") < 30, "18–29")
         .when(F.col("Age") < 45, "30–44")
         .when(F.col("Age") < 60, "45–59")
         .otherwise("60+")
    )
    # BMI banding (rough WHO-style)
    .withColumn(
        "BMI_Band",
        F.when(F.col("BMI").isNull(), "Unknown")
         .when(F.col("BMI") < 18.5, "Underweight")
         .when(F.col("BMI") < 25, "Normal")
         .when(F.col("BMI") < 30, "Overweight")
         .when(F.col("BMI") < 35, "Obese")
         .otherwise("Extremely Obese")
    )
    # Smoker status normalization
    .withColumn(
        "Smoker_Status",
        F.when(F.lower(F.col("Smoker_Flag")).isin("y", "yes", "1"), "Smoker")
         .when(F.lower(F.col("Smoker_Flag")).isin("n", "no", "0"), "Non-Smoker")
         .otherwise("Unknown")
    )
)

fact_enriched_1.select(
    "Member_ID", "Age", "Age_Band", "BMI", "BMI_Band", "Smoker_Flag", "Smoker_Status"
).show(10, truncate=False)


+------------+---+--------+------------------+----------+-----------+-------------+
|Member_ID   |Age|Age_Band|BMI               |BMI_Band  |Smoker_Flag|Smoker_Status|
+------------+---+--------+------------------+----------+-----------+-------------+
|MEM_00000008|56 |45–59   |23.140940890221643|Normal    |Y          |Smoker       |
|MEM_00001714|71 |60+     |27.963719840043765|Overweight|N          |Non-Smoker   |
|MEM_00001746|75 |60+     |30.541100957674118|Obese     |N          |Non-Smoker   |
|MEM_00001967|37 |30–44   |25.982850362358867|Overweight|Y          |Smoker       |
|MEM_00001995|66 |60+     |32.6294573451364  |Obese     |N          |Non-Smoker   |
|MEM_00002251|42 |30–44   |21.91447115325639 |Normal    |Y          |Smoker       |
|MEM_00002445|24 |18–29   |23.72192319873991 |Normal    |N          |Non-Smoker   |
|MEM_00002649|67 |60+     |27.321644871101395|Overweight|Y          |Smoker       |
|MEM_00003068|44 |30–44   |32.15889644505073 |Obese     |Y          |Smoker 

# 🟡 Cell 4 — Feature engineering: chronic, employment, region groupings

In [9]:
# Cell 4 — Feature engineering: Chronic, Employment, Region groupings

fact_enriched_2 = (
    fact_enriched_1
    # Simple chronic disease flag
    .withColumn(
        "Chronic_Flag",
        F.when(F.lower(F.col("Chronic_Disease")) == "none", 0)
         .when(F.col("Chronic_Disease").isNull(), 0)
         .otherwise(1)
    )
    # Coarse chronic disease grouping
    .withColumn(
        "Chronic_Group",
        F.when(F.lower(F.col("Chronic_Disease")).isin("none", "na", "n/a"), "None")
         .when(F.col("Chronic_Disease").isNull(), "None")
         .when(F.lower(F.col("Chronic_Disease")).rlike("diab"), "Diabetes")
         .when(F.lower(F.col("Chronic_Disease")).rlike("hyper"), "Hypertension")
         .when(F.lower(F.col("Chronic_Disease")).rlike("asth"), "Asthma")
         .otherwise("Other condition")
    )
    # Employment grouping
    .withColumn(
        "Employment_Group",
        F.when(F.lower(F.col("Employment_Status")).like("%employ%"), "Employed")
         .when(F.lower(F.col("Employment_Status")).like("%student%"), "Student")
         .when(F.lower(F.col("Employment_Status")).like("%retired%"), "Retired")
         .when(F.lower(F.col("Employment_Status")).like("%unemploy%"), "Unemployed")
         .otherwise("Other")
    )
    # Region grouping (simple numeric buckets or string)
    .withColumn(
        "Region_Group",
        F.when(F.col("Region").isNull(), "Unknown")
         .when(F.col("Region").cast("int") < 50, "Region_0-49")
         .when(F.col("Region").cast("int") < 150, "Region_50-149")
         .when(F.col("Region").cast("int") < 250, "Region_150-249")
         .when(F.col("Region").cast("int") < 350, "Region_250-349")
         .otherwise("Region_350+")
    )
)

fact_enriched_2.select(
    "Member_ID", "Chronic_Disease", "Chronic_Flag", "Chronic_Group",
    "Employment_Status", "Employment_Group",
    "Region", "Region_Group"
).show(10, truncate=False)


+------------+---------------+------------+-------------+-----------------+----------------+------+--------------+
|Member_ID   |Chronic_Disease|Chronic_Flag|Chronic_Group|Employment_Status|Employment_Group|Region|Region_Group  |
+------------+---------------+------------+-------------+-----------------+----------------+------+--------------+
|MEM_00000008|Asthma         |1           |Asthma       |Student          |Student         |280   |Region_250-349|
|MEM_00001714|None           |0           |None         |Employed         |Employed        |360   |Region_350+   |
|MEM_00001746|Diabetes       |1           |Diabetes     |Student          |Student         |80    |Region_50-149 |
|MEM_00001967|Hypertension   |1           |Hypertension |Employed         |Employed        |280   |Region_250-349|
|MEM_00001995|Asthma         |1           |Asthma       |Employed         |Employed        |360   |Region_350+   |
|MEM_00002251|None           |0           |None         |Student          |Stude

# 🟡 Cell 5 — Simple DQ flags (age/BMI validity)

In [10]:
# Cell 5 — Simple DQ flags (age/BMI validity)

fact_members = (
    fact_enriched_2
    .withColumn(
        "dq_age_valid",
        F.when((F.col("Age").isNull()) | ((F.col("Age") >= 0) & (F.col("Age") <= 110)), 1).otherwise(0)
    )
    .withColumn(
        "dq_bmi_valid",
        F.when((F.col("BMI").isNull()) | ((F.col("BMI") >= 10) & (F.col("BMI") <= 60)), 1).otherwise(0)
    )
)

print("fact_members rowcount:", fact_members.count())
fact_members.printSchema()
fact_members.show(5, truncate=False)


fact_members rowcount: 381109
root
 |-- Member_ID: string (nullable = true)
 |-- Policy_ID: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- BMI: double (nullable = true)
 |-- Smoker_Flag: string (nullable = true)
 |-- Chronic_Disease: string (nullable = true)
 |-- Employment_Status: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Age_Band: string (nullable = false)
 |-- BMI_Band: string (nullable = false)
 |-- Smoker_Status: string (nullable = false)
 |-- Chronic_Flag: integer (nullable = false)
 |-- Chronic_Group: string (nullable = false)
 |-- Employment_Group: string (nullable = false)
 |-- Region_Group: string (nullable = false)
 |-- dq_age_valid: integer (nullable = false)
 |-- dq_bmi_valid: integer (nullable = false)



+------------+----------+---+------+------------------+-----------+---------------+-----------------+------+--------+----------+-------------+------------+-------------+----------------+--------------+------------+------------+
|Member_ID   |Policy_ID |Age|Gender|BMI               |Smoker_Flag|Chronic_Disease|Employment_Status|Region|Age_Band|BMI_Band  |Smoker_Status|Chronic_Flag|Chronic_Group|Employment_Group|Region_Group  |dq_age_valid|dq_bmi_valid|
+------------+----------+---+------+------------------+-----------+---------------+-----------------+------+--------+----------+-------------+------------+-------------+----------------+--------------+------------+------------+
|MEM_00000008|P_00000008|56 |F     |23.140940890221643|Y          |Asthma         |Student          |280   |45–59   |Normal    |Smoker       |1           |Asthma       |Student         |Region_250-349|1           |1           |
|MEM_00001714|P_00001714|71 |M     |27.963719840043765|N          |None           |Emplo

# 🟡 Cell 6 — Write Gold fact_members to Delta + register table

In [11]:
# Cell 6 — Write Gold fact_members to Delta + register table

(
    fact_members
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(GOLD_FACT_MEMBERS_PATH)
)

print("✅ Gold fact_members written to:", GOLD_FACT_MEMBERS_PATH)

full_table_name = f"{DB_GOLD}.fact_members"

spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")
spark.sql(f"""
CREATE TABLE {full_table_name}
USING DELTA
LOCATION '{GOLD_FACT_MEMBERS_PATH}'
""")

print("✅ Registered table:", full_table_name)
spark.table(full_table_name).show(5, truncate=False)


✅ Gold fact_members written to: abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_members
✅ Registered table: bupa_gold.fact_members


+------------+----------+---+------+------------------+-----------+---------------+-----------------+------+--------+----------+-------------+------------+-------------+----------------+--------------+------------+------------+
|Member_ID   |Policy_ID |Age|Gender|BMI               |Smoker_Flag|Chronic_Disease|Employment_Status|Region|Age_Band|BMI_Band  |Smoker_Status|Chronic_Flag|Chronic_Group|Employment_Group|Region_Group  |dq_age_valid|dq_bmi_valid|
+------------+----------+---+------+------------------+-----------+---------------+-----------------+------+--------+----------+-------------+------------+-------------+----------------+--------------+------------+------------+
|MEM_00000008|P_00000008|56 |F     |23.140940890221643|Y          |Asthma         |Student          |280   |45–59   |Normal    |Smoker       |1           |Asthma       |Student         |Region_250-349|1           |1           |
|MEM_00001714|P_00001714|71 |M     |27.963719840043765|N          |None           |Emplo

# sample data from "bupa_gold.fact_members"

In [12]:
# Read Gold fact_members table
fact_members = spark.table("bupa_gold.fact_members")

# Display schema and sample data
fact_members.printSchema()
fact_members.show(10, truncate=False)
print("Row count:", fact_members.count())

root
 |-- Member_ID: string (nullable = true)
 |-- Policy_ID: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- BMI: double (nullable = true)
 |-- Smoker_Flag: string (nullable = true)
 |-- Chronic_Disease: string (nullable = true)
 |-- Employment_Status: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Age_Band: string (nullable = true)
 |-- BMI_Band: string (nullable = true)
 |-- Smoker_Status: string (nullable = true)
 |-- Chronic_Flag: integer (nullable = true)
 |-- Chronic_Group: string (nullable = true)
 |-- Employment_Group: string (nullable = true)
 |-- Region_Group: string (nullable = true)
 |-- dq_age_valid: integer (nullable = true)
 |-- dq_bmi_valid: integer (nullable = true)



+------------+----------+---+------+------------------+-----------+---------------+-----------------+------+--------+----------+-------------+------------+-------------+----------------+--------------+------------+------------+
|Member_ID   |Policy_ID |Age|Gender|BMI               |Smoker_Flag|Chronic_Disease|Employment_Status|Region|Age_Band|BMI_Band  |Smoker_Status|Chronic_Flag|Chronic_Group|Employment_Group|Region_Group  |dq_age_valid|dq_bmi_valid|
+------------+----------+---+------+------------------+-----------+---------------+-----------------+------+--------+----------+-------------+------------+-------------+----------------+--------------+------------+------------+
|MEM_00000008|P_00000008|56 |F     |23.140940890221643|Y          |Asthma         |Student          |280   |45–59   |Normal    |Smoker       |1           |Asthma       |Student         |Region_250-349|1           |1           |
|MEM_00001714|P_00001714|71 |M     |27.963719840043765|N          |None           |Emplo

# 🟡 Cell 7 — Quick profiling: demographics & risk mix

In [13]:
# Cell 7 — Quick profiling: demographics & risk mix

print("=== Members by Age_Band & Gender ===")
(
    fact_members.groupBy("Age_Band", "Gender")
    .count()
    .orderBy("Age_Band", "Gender")
    .show(truncate=False)
)

print("=== Smokers vs Non-Smokers by Age_Band ===")
(
    fact_members.groupBy("Age_Band", "Smoker_Status")
    .count()
    .orderBy("Age_Band", "Smoker_Status")
    .show(truncate=False)
)

print("=== Chronic condition mix by Employment_Group ===")
(
    fact_members.groupBy("Employment_Group", "Chronic_Group")
    .count()
    .orderBy("Employment_Group", "Chronic_Group")
    .show(truncate=False)
)


=== Members by Age_Band & Gender ===


+--------+------+-----+
|Age_Band|Gender|count|
+--------+------+-----+
|18–29   |F     |86098|
|18–29   |M     |69105|
|30–44   |F     |40651|
|30–44   |M     |53307|
|45–59   |F     |29815|
|45–59   |M     |55360|
|60+     |F     |18456|
|60+     |M     |28317|
+--------+------+-----+

=== Smokers vs Non-Smokers by Age_Band ===


+--------+-------------+------+
|Age_Band|Smoker_Status|count |
+--------+-------------+------+
|18–29   |Non-Smoker   |108460|
|18–29   |Smoker       |46743 |
|30–44   |Non-Smoker   |35338 |
|30–44   |Smoker       |58620 |
|45–59   |Non-Smoker   |26856 |
|45–59   |Smoker       |58319 |
|60+     |Non-Smoker   |18042 |
|60+     |Smoker       |28731 |
+--------+-------------+------+

=== Chronic condition mix by Employment_Group ===
+----------------+-------------+-----+
|Employment_Group|Chronic_Group|count|
+----------------+-------------+-----+
|Employed        |Asthma       |50212|
|Employed        |Diabetes     |26856|
|Employed        |Hypertension |53106|
|Employed        |None         |93364|
|Retired         |Asthma       |25927|
|Retired         |Diabetes     |13841|
|Retired         |Hypertension |27421|
|Retired         |None         |48083|
|Student         |Asthma       |9514 |
|Student         |Diabetes     |5048 |
|Student         |Hypertension |10086|
|Student         |N

# 🟡 Cell 8 — Region & BMI distribution

In [14]:
# Cell 8 — Region & BMI distribution

print("=== Members by Region_Group ===")
(
    fact_members.groupBy("Region_Group")
    .count()
    .orderBy("Region_Group")
    .show(truncate=False)
)

print("=== BMI_Band vs Smoker_Status ===")
(
    fact_members.groupBy("BMI_Band", "Smoker_Status")
    .count()
    .orderBy("BMI_Band", "Smoker_Status")
    .show(truncate=False)
)


=== Members by Region_Group ===
+--------------+------+
|Region_Group  |count |
+--------------+------+
|Region_0-49   |18119 |
|Region_150-249|36505 |
|Region_250-349|151649|
|Region_350+   |101502|
|Region_50-149 |73334 |
+--------------+------+

=== BMI_Band vs Smoker_Status ===
+----------+-------------+-----+
|BMI_Band  |Smoker_Status|count|
+----------+-------------+-----+
|Normal    |Non-Smoker   |55023|
|Normal    |Smoker       |55863|
|Obese     |Non-Smoker   |55015|
|Obese     |Smoker       |56142|
|Overweight|Non-Smoker   |78658|
|Overweight|Smoker       |80408|
+----------+-------------+-----+



# 🟡 Cell 9 — Metrics logging (for observability)

In [15]:
# Cell 9 — Metrics logging (for observability)

# Reuse utils_silver.write_metric — current signature: (spark, name, value, context, paths_silver)

write_metric(
    spark=spark,
    name="rowcount_fact_members",
    value=fact_members.count(),
    context="fact_members_gold",
    paths_silver=paths_silver
)

write_metric(
    spark=spark,
    name="distinct_member_ids_fact_members",
    value=fact_members.select("Member_ID").distinct().count(),
    context="fact_members_gold",
    paths_silver=paths_silver
)

metrics_df = spark.read.format("delta").load(paths_silver["_metrics"])
metrics_df.orderBy(F.col("ts").desc()).show(20, truncate=False)


[METRIC] rowcount_fact_members=381109 ctx=fact_members_gold


[METRIC] distinct_member_ids_fact_members=381109 ctx=fact_members_gold


+---------------------------------+------+------------------+--------------------------+
|metric                           |value |context           |ts                        |
+---------------------------------+------+------------------+--------------------------+
|distinct_member_ids_fact_members |381109|fact_members_gold |2025-12-08 23:10:45.084442|
|rowcount_fact_members            |381109|fact_members_gold |2025-12-08 23:10:35.810882|
|distinct_claim_ids_fact_claims   |558211|gold_fact_claims  |2025-12-08 23:03:04.95712 |
|rowcount_fact_claims             |558211|gold_fact_claims  |2025-12-08 23:02:55.967222|
|distinct_member_ids_fact_members |381109|fact_members_gold |2025-12-05 12:28:49.329867|
|rowcount_fact_members            |381109|fact_members_gold |2025-12-05 12:28:41.911751|
|distinct_policy_ids_fact_policies|381109|fact_policies_gold|2025-12-05 12:26:53.739201|
|rowcount_fact_policies           |381109|fact_policies_gold|2025-12-05 12:26:45.024747|
|distinct_claim_ids_f

# 🟡 Cell 10 — Final sanity check from table

In [16]:
# Cell 10 — Final sanity check from the registered Gold table

full_table_name = f"{DB_GOLD}.fact_members"
gold_members = spark.table(full_table_name)

print("Gold fact_members rowcount:", gold_members.count())
gold_members.show(10, truncate=False)

# Optional: simple check that Age_Band / BMI_Band exist
print("Columns in Gold fact_members:")
print(gold_members.columns)

print("✅ Gold fact_members ready for BI / ML.")


Gold fact_members rowcount: 381109


+------------+----------+---+------+------------------+-----------+---------------+-----------------+------+--------+----------+-------------+------------+-------------+----------------+--------------+------------+------------+
|Member_ID   |Policy_ID |Age|Gender|BMI               |Smoker_Flag|Chronic_Disease|Employment_Status|Region|Age_Band|BMI_Band  |Smoker_Status|Chronic_Flag|Chronic_Group|Employment_Group|Region_Group  |dq_age_valid|dq_bmi_valid|
+------------+----------+---+------+------------------+-----------+---------------+-----------------+------+--------+----------+-------------+------------+-------------+----------------+--------------+------------+------------+
|MEM_00000008|P_00000008|56 |F     |23.140940890221643|Y          |Asthma         |Student          |280   |45–59   |Normal    |Smoker       |1           |Asthma       |Student         |Region_250-349|1           |1           |
|MEM_00001714|P_00001714|71 |M     |27.963719840043765|N          |None           |Emplo

# 🟦 A) Gold Fact Members — Schema & Rowcount Audit

In [13]:
# A) Gold Fact Members — Schema & Rowcount Audit

gold_members = spark.table("bupa_gold.fact_members")

print("Rowcount:", gold_members.count())
gold_members.printSchema()

from pyspark.sql.functions import col

print("\nNull counts per column:")
nulls = [
    (c, gold_members.filter(col(c).isNull()).count())
    for c in gold_members.columns
]
for c, n in nulls:
    print(f"{c}: {n}")


Rowcount: 381109
root
 |-- Member_ID: string (nullable = true)
 |-- Policy_ID: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- BMI: double (nullable = true)
 |-- Smoker_Flag: string (nullable = true)
 |-- Chronic_Disease: string (nullable = true)
 |-- Employment_Status: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Age_Band: string (nullable = true)
 |-- BMI_Band: string (nullable = true)
 |-- Smoker_Status: string (nullable = true)
 |-- Chronic_Flag: integer (nullable = true)
 |-- Chronic_Group: string (nullable = true)
 |-- Employment_Group: string (nullable = true)
 |-- Region_Group: string (nullable = true)
 |-- dq_age_valid: integer (nullable = true)
 |-- dq_bmi_valid: integer (nullable = true)


Null counts per column:
Member_ID: 0
Policy_ID: 0
Age: 0
Gender: 0
BMI: 0
Smoker_Flag: 0
Chronic_Disease: 0
Employment_Status: 0
Region: 0
Age_Band: 0
BMI_Band: 0
Smoker_Status: 0
Chronic_Flag: 0
Chronic_Group: 

This validates:

Schema matches design

No unexpected nulls

Fully loaded dataset

🟦 B) Compare Silver Members → Gold Members

This proves that Gold did not lose or change base fields, only enriched them.

In [17]:
# B) Compare Silver Members → Gold Members

silver = spark.read.format("delta").load(paths_silver["members"])
gold = spark.table("bupa_gold.fact_members")

silver_cols = set(silver.columns)
gold_cols   = set(gold.columns)

print("Columns added in Gold:", sorted(gold_cols - silver_cols))
print("Columns dropped in Gold:", sorted(silver_cols - gold_cols))
print("Common columns:", len(silver_cols & gold_cols))


Columns added in Gold: ['BMI_Band', 'Chronic_Flag', 'Chronic_Group', 'Employment_Group', 'Region_Group', 'Smoker_Flag', 'Smoker_Status']
Columns dropped in Gold: ['BMI_Category', 'Risk_Segment', 'Smoker']
Common columns: 11


Expected output:

Added: Age_Band, BMI_Band, Smoker_Status, Chronic_Flag, Chronic_Group, Employment_Group, Region_Group, dq_age_valid, dq_bmi_valid

Dropped: none

Common: 9

Perfect for interviews.

# 🟦 C) Gold Members — KPI Profiling

This is the dashboard-ready metrics for BI teams.

# 1️⃣ Age distribution

In [18]:
print("1️⃣ Age distribution")
gold_members.groupBy("Age_Band").count().orderBy("Age_Band").show(truncate=False)

print("2️⃣ BMI distribution")
gold_members.groupBy("BMI_Band").count().orderBy("BMI_Band").show(truncate=False)

print("3️⃣ Smokers vs Non-Smokers")
gold_members.groupBy("Smoker_Status").count().show()

print("4️⃣ Chronic condition impact")
gold_members.groupBy("Chronic_Group").count().orderBy("Chronic_Group").show()

print("5️⃣ Employment status segments")
gold_members.groupBy("Employment_Group").count().orderBy("Employment_Group").show()


print("These 5 KPIs are what BI teams, underwriting, and actuaries need.")


1️⃣ Age distribution


+--------+------+
|Age_Band|count |
+--------+------+
|18–29   |155203|
|30–44   |93958 |
|45–59   |85175 |
|60+     |46773 |
+--------+------+

2️⃣ BMI distribution


+----------+------+
|BMI_Band  |count |
+----------+------+
|Normal    |110886|
|Obese     |111157|
|Overweight|159066|
+----------+------+

3️⃣ Smokers vs Non-Smokers


+-------------+------+
|Smoker_Status| count|
+-------------+------+
|   Non-Smoker|188696|
|       Smoker|192413|
+-------------+------+

4️⃣ Chronic condition impact
+-------------+------+
|Chronic_Group| count|
+-------------+------+
|       Asthma| 85653|
|     Diabetes| 45745|
| Hypertension| 90613|
|         None|159098|
+-------------+------+

5️⃣ Employment status segments
+----------------+------+
|Employment_Group| count|
+----------------+------+
|        Employed|223538|
|         Retired|115272|
|         Student| 42299|
+----------------+------+

These 5 KPIs are what BI teams, underwriting, and actuaries need.


# 🟦 D) Relationship analysis: Members ↔ Policies / Claims

In [16]:
print("1️⃣ Members per Policy")
gold_members.groupBy("Policy_ID").count().orderBy(F.col("count").desc()).show(10)


print("Shows which policies cover multiple members.")
print("2️⃣ High BMI vs Smoker interaction")
gold_members.groupBy("BMI_Band", "Smoker_Status").count().orderBy("BMI_Band").show()


print("Risk modelling KPI.")

print("3️⃣ Region vs Chronic Disease")
gold_members.groupBy("Region_Group", "Chronic_Group").count().show()


print("Useful for actuaries & health analytics.")

1️⃣ Members per Policy


+----------+-----+
| Policy_ID|count|
+----------+-----+
|P_00000229|    1|
|P_00000013|    1|
|P_00000450|    1|
|P_00000199|    1|
|P_00000457|    1|
|P_00000240|    1|
|P_00000500|    1|
|P_00000276|    1|
|P_00000516|    1|
|P_00000441|    1|
+----------+-----+
only showing top 10 rows

Shows which policies cover multiple members.
2️⃣ High BMI vs Smoker interaction
+----------+-------------+-----+
|  BMI_Band|Smoker_Status|count|
+----------+-------------+-----+
|    Normal|   Non-Smoker|55023|
|    Normal|       Smoker|55863|
|     Obese|       Smoker|56142|
|     Obese|   Non-Smoker|55015|
|Overweight|       Smoker|80408|
|Overweight|   Non-Smoker|78658|
+----------+-------------+-----+

Risk modelling KPI.
3️⃣ Region vs Chronic Disease
+--------------+-------------+-----+
|  Region_Group|Chronic_Group|count|
+--------------+-------------+-----+
| Region_50-149|       Asthma|16607|
|Region_150-249|     Diabetes| 4375|
|Region_250-349| Hypertension|36259|
|Region_250-349|       As

# 🟦 Gold Members — Suggested KPIs (You got some already)

In [17]:
print("1️⃣ Smoker ratio")
gold_members.groupBy("Smoker_Status").count().show()

print("2️⃣ Chronic disease distribution")
gold_members.groupBy("Chronic_Group").count().show()

print("3️⃣ High-risk segments (Smoker + Obese + Chronic)")
gold_members.filter(
    (F.col("Smoker_Status") == "Smoker") &
    (F.col("BMI_Band").isin("Obese", "Extremely Obese")) &
    (F.col("Chronic_Flag") == 1)
).count()

print("4️⃣ Regional heatmap (BI team uses)")
(
    gold_members.groupBy("Region_Group")
    .agg(F.count("*").alias("n_members"))
    .orderBy("Region_Group")
    .show()
)

1️⃣ Smoker ratio
+-------------+------+
|Smoker_Status| count|
+-------------+------+
|   Non-Smoker|188696|
|       Smoker|192413|
+-------------+------+

2️⃣ Chronic disease distribution
+-------------+------+
|Chronic_Group| count|
+-------------+------+
|         None|159098|
|     Diabetes| 45745|
| Hypertension| 90613|
|       Asthma| 85653|
+-------------+------+

3️⃣ High-risk segments (Smoker + Obese + Chronic)


4️⃣ Regional heatmap (BI team uses)
+--------------+---------+
|  Region_Group|n_members|
+--------------+---------+
|   Region_0-49|    18119|
|Region_150-249|    36505|
|Region_250-349|   151649|
|   Region_350+|   101502|
| Region_50-149|    73334|
+--------------+---------+

